In [1]:
from pynq import Overlay
from pynq import allocate
import numpy as np

slowOverlay = Overlay("/root/jupyter_notebooks/matMulSlow/slowMult.bit")

print("available IPs:", slowOverlay.ip_dict.keys())
print("Overlay loaded.")

slowMultIP = slowOverlay.matrixMult_0


available IPs: dict_keys(['matrixMult_0', 'zynq_ultra_ps_e_0'])
Overlay loaded.


In [2]:
#get the register map
print(slowMultIP.register_map)


RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  A_1 = Register(A=write-only),
  A_2 = Register(A=write-only),
  B_1 = Register(B=write-only),
  B_2 = Register(B=write-only),
  C_1 = Register(C=write-only),
  C_2 = Register(C=write-only)
}


In [3]:
N = 16
# allocate memory for 3 16x16 matrices stored in ddr
A = allocate(shape=(N, N), dtype=np.int32)
B = allocate(shape=(N, N), dtype=np.int32)
C = allocate(shape=(N, N), dtype=np.int32)


# initializes A and B with random integers
A[:] = np.random.randint(-5, 5, (N, N))
B[:] = np.random.randint(-5, 5, (N, N))

print(A, "\n\n", B)
#print(B)

# save IP's AXI-Lite Registers
regMap = slowMultIP.register_map

# mask in addresses for A,B,C
# needs to be masked because CPU uses 64 bit addressing, but AXI-lite is only 32 bit wide
regMap.A_1.A = A.physical_address & 0xFFFFFFFF
regMap.A_2.A = (A.physical_address >> 32) & 0xFFFFFFFF

regMap.B_1.B = B.physical_address & 0xFFFFFFFF
regMap.B_2.B = (B.physical_address >> 32) & 0xFFFFFFFF

regMap.C_1.C = C.physical_address & 0xFFFFFFFF
regMap.C_2.C = (C.physical_address >> 32) & 0xFFFFFFFF


[[ 1 -1 -3  1 -1  3 -3  0 -3  4 -5  1  4 -5  1 -4]
 [-2 -3  0 -2  1  2 -5  1  0  4 -2 -3 -4 -3 -3  3]
 [-2 -3 -5 -1 -5 -4  3  4 -2 -5 -5  3  3 -1 -2 -3]
 [ 0 -1 -5  2 -5 -4 -1  2 -1  3  2 -3 -2 -5 -2 -1]
 [ 2  1  3  3  0  1  4  2 -5  3 -3  2 -2  2 -4  2]
 [ 4  2 -4  4  1  0 -5 -5 -3 -5 -3 -1  3 -5 -2 -4]
 [ 2 -2 -2 -2  3 -3 -3 -3 -3  4  2  2  1  1  0  3]
 [-3  2 -4  4 -3 -5 -3 -1  4  0 -4 -2 -2 -3  0 -3]
 [-5 -3 -4  1 -3  2 -5  0  2 -5 -4 -3  3 -4 -2  4]
 [-3 -2 -3  3 -3  1  4  1  4  3  2 -1  4 -4 -1  0]
 [-5  3 -5 -4  2 -4  0  1  0 -2  3  2 -5 -3  1 -3]
 [-3 -5  4 -1  2  0  4 -1 -4 -3 -2 -4 -3 -4  0 -2]
 [ 3 -4 -2  3  1  2 -3 -2 -4  4  2 -2 -3  2 -2  1]
 [ 3 -4 -1 -1 -1  1 -3  1  3  0 -1  1  3  3  1  3]
 [-1  2  3 -5  2  3 -2  2  3  0 -2 -3 -2 -3  0 -5]
 [-1  0  4  1  0 -1  2 -1  2 -3 -1 -2 -4 -5  2  0]] 

 [[ 1 -4 -5 -3  1  2  2  3  4 -1  1 -2  1 -4  0  2]
 [-2 -4  2  0 -4 -4  2  0  0 -5 -1 -1  1 -2  1  0]
 [-5 -3 -4 -1  1 -1 -2  3  4 -2 -5  4 -5 -5 -5 -1]
 [ 4  4  4 -2 -1 -5  2  1 -

In [4]:
from time import perf_counter

# record start time
startTime = perf_counter()

# starts Matrix Accelerator
slowMultIP.register_map.CTRL.AP_START = 1

# wait until FPGA finishes (poll the done bit)
while slowMultIP.register_map.CTRL.AP_DONE == 0:
    pass

# record end time and calculate total time
endTime = perf_counter()
totalTime = endTime - startTime

print("Hardware multiply finished!")

Hardware multiply finished!


In [5]:
# create the baseline reference and compare results
C_ref = A @ B

print(C_ref, "\n\n", C)
print("Max error:", np.max(np.abs(C_ref - C)))
print("Correct:", np.array_equal(C_ref, C))


[[ -6  45  42 -40 -29   0  27 -32  -1  64  14  53  16  60  -1 -27]
 [-29  39  -1 -23  30  25  18  23 -23   3  26  18  19   1   7   2]
 [ 97  61  74  50  10  19  30 -28  -6 110  45  17 -35  90  58 -34]
 [ 68  38  62  20  49 -32  59 -36 -22  63  52  30  36  45  53 -20]
 [ 20  -3  -1  -8  32  -4  40  -1 -12 -18  58  21 -23 -94  21  -6]
 [ 60  59  67 -12 -35  15  -6  -5  13  97   3  77  94  15  21  21]
 [ -4  -6 -31 -11  -1   6  -7 -36  -1   2  56 -20  36 -23 -36  51]
 [ 46  -1  64  -6  -2 -54  46  -3 -53  54  30  28  60  31  78 -32]
 [ 28 111  95 -38 -54  35  -4  39   6  63  11  41  12  83  68  16]
 [ 33  19  49 -14   1 -30  20 -30 -35  14   1   8 -29  58  35 -17]
 [ 36  24   6 110  26 -37  -8 -43 -37  37 -13 -51  54  64 -17 -48]
 [ 28  21  20  60  63  42 -57   1 -25  24 -45  64  11 -20 -18 -23]
 [  5  28  -8 -29  39  25  14 -15 -11   5  53  25  51 -51   4  34]
 [-25   3 -31 -59 -32  32   6  35  35  -2  21 -34 -31  21   7  30]
 [-63 -31 -25  20  22  25 -14  27  -9 -14 -77  13 -20  38 -39 

In [6]:
def numpyVSaccel( ip, A, B, C, nTests=50 ):
    numpy_times = []
    accel_times = []
    all_correct = True
    for i in range(nTests):
        # random test case
        A_np = np.random.randint(-100, 100, (N, N))
        B_np = np.random.randint(-100, 100, (N, N))
        
        # NumPy timing
        startTime = perf_counter()
        C_ref = A_np @ B_np
        endTime = perf_counter()
        numpy_times.append(endTime - startTime)
        
        # copy to FPGA buffers
        A[:] = A_np
        B[:] = B_np
        
        # run accelerator
        regMap = ip.register_map

        regMap.A_1.A = A.physical_address & 0xFFFFFFFF
        regMap.A_2.A = (A.physical_address >> 32) & 0xFFFFFFFF

        regMap.B_1.B = B.physical_address & 0xFFFFFFFF
        regMap.B_2.B = (B.physical_address >> 32) & 0xFFFFFFFF

        regMap.C_1.C = C.physical_address & 0xFFFFFFFF
        regMap.C_2.C = (C.physical_address >> 32) & 0xFFFFFFFF
        
        # accelerator timing
        startTime = perf_counter()
        ip.register_map.CTRL.AP_START = 1
        while ip.register_map.CTRL.AP_DONE == 0:
            pass
        endTime = perf_counter()
        accel_times.append(endTime - startTime)
        
        # verify
        if not np.array_equal(C_ref, C):
            all_correct = False
            print(f"ERROR: Mismatch on trial {i}")
            break
        
    # print the metrics
    print("Correct for all trials:", all_correct)
    print(f"Avg NumPy time:   {np.mean(numpy_times)*1e6:.2f} us")
    print(f"Avg Accel time:   {np.mean(accel_times)*1e6:.2f} us")
    print(f"Speedup:          {np.mean(numpy_times) / np.mean(accel_times):.2f}x")

    return numpy_times, accel_times

In [10]:
# slow results
print("Slow Matrix Multiplication Implementation")
numpy_times_slow, accel_times_slow = numpyVSaccel(slowMultIP, A, B, C, 50)

Slow Matrix Multiplication Implementation
Correct for all trials: True
Avg NumPy time:   48.27 us
Avg Accel time:   65.39 us
Speedup:          0.74x


In [12]:
# load fast overlay
fastOverlay = Overlay("/root/jupyter_notebooks/matMulSlow/fastMult.bit")
print("available IPs:", fastOverlay.ip_dict.keys())
fastMultIP = fastOverlay.matrixMultFast_0

#get the register map
#print(slowMultIP.register_map)

# fast results
print("Fast Matrix Multiplication Implementation")
numpy_times_fast, accel_times_fast = numpyVSaccel(fastMultIP, A, B, C, 50)

available IPs: dict_keys(['matrixMultFast_0', 'zynq_ultra_ps_e_0'])
Fast Matrix Multiplication Implementation
Correct for all trials: True
Avg NumPy time:   46.28 us
Avg Accel time:   61.48 us
Speedup:          0.75x
